In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch

from PIL import Image
from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
)

**Setup and model load**

In [2]:
MODEL_DIR = OUTPUT_DIR = "/content/drive/MyDrive/Fine tuned models/vit_beans_best"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = AutoImageProcessor.from_pretrained(MODEL_DIR)
model = AutoModelForImageClassification.from_pretrained(
    MODEL_DIR
).to(device)

model.eval()

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (o_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layernorm_before): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (layernorm_after): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (mlp): ViTMLP(
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out

In [3]:
dataset = load_dataset("AI-Lab-Makerere/beans")

print("Device:", device)
print("Classes:", model.config.id2label)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Device: cuda
Classes: {0: 'angular_leaf_spot', 1: 'bean_rust', 2: 'healthy'}


**Prediction function**

In [4]:
def predict(image):
    inputs = processor(
        images=image.convert("RGB"),
        return_tensors="pt"
    )
    pixel_values = inputs["pixel_values"].to(device)
    with torch.inference_mode():
        logits = model(pixel_values=pixel_values).logits
    
    probabilities = torch.softmax(logits, dim=-1)[0]
    predicted_id = probabilities.argmax().item()

    return probabilities.cpu(), predicted_id

**Patch-occlusion explanation**

In [ ]:
def create_occlusion_map(
    image, target_class,
    patch_size=16, batch_size=32
):
    image = image.convert("RGB").resize((224, 224))
    image_array = np.array(image)

    baseline_probabilities, _ = predict(image)
    baseline_score = baseline_probabilities[target_class].item()
    occluded_images = []

    for y in range(0, 224, patch_size):
        for x in range(0, 224, patch_size):
            occluded = image_array.copy()

            occluded[y:y + patch_size, x:x + patch_size] = 128
            occluded_images.append(Image.fromarray(occluded))

    occluded_scores = []
    for start in range(0, len(occluded_images), batch_size):
        image_batch = occluded_images[
            start:start + batch_size
        ]

        pixel_values = processor(
            images=image_batch,
            return_tensors="pt",
        )["pixel_values"].to(device)

        with torch.inference_mode():
            logits = model(pixel_values=pixel_values).logits

        probabilities = torch.softmax(logits, dim=-1)
        occluded_scores.extend(
            probabilities[:, target_class].cpu().tolist()
        )

    importance = baseline_score - np.array(occluded_scores)
    importance = np.maximum(importance, 0)

    grid_size = 224 // patch_size
    importance = importance.reshape(grid_size, grid_size)

    if importance.max() > 0:
        importance = (importance / importance.max())

    heatmap = Image.fromarray(np.uint8(importance * 255)).resize(
        (224, 224),
        Image.Resampling.BILINEAR,
    )

    return image, np.array(heatmap) / 255.0

In [6]:
SAMPLE_INDEX = 0

sample = dataset["test"][SAMPLE_INDEX]

image = sample["image"].convert("RGB")
true_id = sample["labels"]
true_label = model.config.id2label[true_id]

probabilities, predicted_id = predict(image)

predicted_label = model.config.id2label[
    predicted_id
]

confidence = probabilities[predicted_id].item()

top_scores, top_indices = torch.topk(
    probabilities,
    k=3,
)

print("Ground truth:", true_label)
print("Prediction  :", predicted_label)
print(f"Confidence  : {confidence:.2%}")

print("\nAll class probabilities:")

for class_id, score in zip(
    top_indices.tolist(),
    top_scores.tolist(),
):
    print(
        f"{model.config.id2label[class_id]:<22}"
        f"{score:.2%}"
    )

resized_image, heatmap = create_occlusion_map(
    image=image,
    target_class=predicted_id,
)

Ground truth: angular_leaf_spot
Prediction  : angular_leaf_spot
Confidence  : 41.16%

All class probabilities:
angular_leaf_spot     41.16%
bean_rust             36.82%
healthy               22.02%
